# エチオピアコーヒー適地マッピング
## ― 衛星データで「コーヒー2050年問題」に挑む ―

**Frontier Academy Fukuoka 卒業制作（2026年6月）**  
**作成者：** 森山光明（Mitsuaki Moriyama）

---

## 🎯 このノートブックの目的

アラビカコーヒーの世界的産地であるエチオピアを対象に、複数の衛星データを統合して**現状の栽培適地を可視化**する。  
既知の三大産地（Sidamo / Yirgacheffe / Harrar）との一致を検証し、「コーヒー2050年問題」への入り口とする。

## 📊 使用データ（すべてGoogle Earth Engine経由）

| データ | プロダクト | 内容 |
|---|---|---|
| 標高 | SRTM (USGS) | 解像度30m DEM |
| 気温 | ERA5-Land | 月別気温 → 年平均 |
| 降水量 | CHIRPS | 日降水量 → 年合計 |
| 植生 | Sentinel-2 SR | NDVI（光合成活性の指標） |

## 🌱 アラビカ種の適地条件（既知の農学知見）

| 要素 | 最適範囲 | 許容範囲 |
|---|---|---|
| 標高 | 1500〜2200m | 1200〜2500m |
| 年平均気温 | 18〜22℃ | 15〜24℃ |
| 年降水量 | 1200〜1800mm | 1000〜2000mm |
| NDVI | 0.4以上 | 0.3以上 |

出典：ICO（国際コーヒー機関）, DaMatta et al. (2007), Bunn et al. (2015)


---
## 1. 環境セットアップ

Google Earth Engine（GEE）の認証が必要。初回のみ `ee.Authenticate()` を実行する。

In [ ]:
# 必要なライブラリのインポート
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 初回のみ認証（コメント外して実行）
# ee.Authenticate()

# 自分のGoogle Cloud プロジェクトIDに置き換える
ee.Initialize(project='your-gee-project-id')

print('✅ Google Earth Engine 初期化完了')

---
## 2. 対象エリアの定義：エチオピア国境

FAO GAUL（行政境界データ）からエチオピアのポリゴンを取得する。

In [ ]:
# エチオピア国境を取得
countries = ee.FeatureCollection('FAO/GAUL/2015/level0')
ethiopia = countries.filter(ee.Filter.eq('ADM0_NAME', 'Ethiopia'))
ethiopia_geom = ethiopia.geometry()

print('✅ エチオピアの境界データ取得完了')
print(f'面積（概算）: {ethiopia_geom.area().getInfo() / 1e6:,.0f} km²')

---
## 3. データレイヤーの取得

### 3-1. 標高（SRTM DEM）

In [ ]:
# SRTMから標高データ取得
elevation = ee.Image('USGS/SRTMGL1_003').clip(ethiopia_geom)

print('✅ 標高データ取得完了')

### 3-2. 年平均気温（ERA5-Land）

直近5年分（2020-2024）の平均を用いて、年平均気温を算出する。  
ERA5の気温はケルビン（K）単位なので、273.15を引いて摂氏に変換する。

In [ ]:
# ERA5-Land 月別気温（2020-2024）
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR') \
    .filterDate('2020-01-01', '2025-01-01') \
    .select('temperature_2m')

# 5年間の平均気温（K → ℃）
temp_celsius = era5.mean().subtract(273.15).clip(ethiopia_geom).rename('temp_c')

print('✅ 年平均気温データ作成完了（2020-2024平均）')

### 3-3. 年降水量（CHIRPS）

In [ ]:
# CHIRPS 日別降水量 → 5年間の年平均降水量
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
    .filterDate('2020-01-01', '2025-01-01') \
    .select('precipitation')

# 年合計を5年分平均化（日降水量を365倍する近似）
annual_precip = chirps.mean().multiply(365).clip(ethiopia_geom).rename('precip_mm')

print('✅ 年降水量データ作成完了（2020-2024平均）')

### 3-4. NDVI（Sentinel-2）

雲なしの良質な観測のみを抽出し、直近1年のNDVI中央値を計算する。

In [ ]:
# 雲マスク関数
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(cirrus_bit).eq(0))
    return image.updateMask(mask).divide(10000)

# Sentinel-2 SR（直近1年）
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterDate('2024-01-01', '2025-01-01') \
    .filterBounds(ethiopia_geom) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(mask_s2_clouds)

# NDVI 中央値
ndvi = s2.median().normalizedDifference(['B8', 'B4']).clip(ethiopia_geom).rename('ndvi')

print('✅ NDVI算出完了')

---
## 4. 適地スコアの算出

各指標を「適地条件」に基づいて0〜1のスコアに変換する。  
**最適範囲＝1.0、許容範囲＝0.5、範囲外＝0** の3段階閾値で実装する。

In [ ]:
# 標高スコア
elev_score = ee.Image(0) \
    .where(elevation.gte(1200).And(elevation.lt(1500)), 0.5) \
    .where(elevation.gte(1500).And(elevation.lte(2200)), 1.0) \
    .where(elevation.gt(2200).And(elevation.lte(2500)), 0.5) \
    .rename('elev_score')

# 気温スコア
temp_score = ee.Image(0) \
    .where(temp_celsius.gte(15).And(temp_celsius.lt(18)), 0.5) \
    .where(temp_celsius.gte(18).And(temp_celsius.lte(22)), 1.0) \
    .where(temp_celsius.gt(22).And(temp_celsius.lte(24)), 0.5) \
    .rename('temp_score')

# 降水量スコア
precip_score = ee.Image(0) \
    .where(annual_precip.gte(1000).And(annual_precip.lt(1200)), 0.5) \
    .where(annual_precip.gte(1200).And(annual_precip.lte(1800)), 1.0) \
    .where(annual_precip.gt(1800).And(annual_precip.lte(2000)), 0.5) \
    .rename('precip_score')

# NDVIスコア（植生があるかどうか）
ndvi_score = ee.Image(0) \
    .where(ndvi.gte(0.3).And(ndvi.lt(0.4)), 0.5) \
    .where(ndvi.gte(0.4), 1.0) \
    .rename('ndvi_score')

print('✅ 各スコア算出完了')

### 4-1. 総合適地スコア（重み付き合成）

コーヒーの生育において、**気温と標高が最も重要**で、降水量とNDVIは補助的と考え、以下の重みで合成する。

| 要素 | 重み | 理由 |
|---|---|---|
| 標高 | 0.30 | 昼夜寒暖差・品質に直結 |
| 気温 | 0.30 | 生育の基本条件 |
| 降水量 | 0.25 | 灌漑の有無に依存 |
| NDVI | 0.15 | 実際の植生有無の検証 |

In [ ]:
# 総合適地スコア
suitability = elev_score.multiply(0.30) \
    .add(temp_score.multiply(0.30)) \
    .add(precip_score.multiply(0.25)) \
    .add(ndvi_score.multiply(0.15)) \
    .rename('suitability')

print('✅ 総合適地スコア算出完了（0.0〜1.0）')

---
## 5. 可視化

geemap によるインタラクティブマップで結果を確認する。

In [ ]:
# マップ初期化（エチオピア中心）
Map = geemap.Map(center=[9.145, 40.4897], zoom=6)

# 標高
Map.addLayer(elevation, {'min': 0, 'max': 4000, 'palette': ['white', 'yellow', 'orange', 'red', 'brown']}, '標高(m)', False)

# 気温
Map.addLayer(temp_celsius, {'min': 10, 'max': 30, 'palette': ['blue', 'cyan', 'yellow', 'orange', 'red']}, '年平均気温(℃)', False)

# 降水量
Map.addLayer(annual_precip, {'min': 0, 'max': 2500, 'palette': ['white', 'lightblue', 'blue', 'darkblue']}, '年降水量(mm)', False)

# NDVI
Map.addLayer(ndvi, {'min': 0, 'max': 0.8, 'palette': ['white', 'yellow', 'green', 'darkgreen']}, 'NDVI', False)

# 総合適地スコア（メイン）
suitability_vis = {
    'min': 0,
    'max': 1,
    'palette': ['white', 'lightyellow', 'yellow', 'orange', 'red', 'darkred']
}
Map.addLayer(suitability, suitability_vis, '☕ コーヒー適地スコア', True)

# エチオピア国境
Map.addLayer(ethiopia.style(color='black', fillColor='00000000', width=2), {}, '国境')

Map

---
## 6. 既知産地での検証

エチオピアの三大コーヒー産地で、適地スコアが本当に高くなっているかを検証する。

| 産地 | 緯度 | 経度 | 特徴 |
|---|---|---|---|
| Sidamo | 6.77 | 38.50 | 柑橘系の華やかな酸味 |
| Yirgacheffe | 6.16 | 38.20 | 花のような香り |
| Harrar | 9.31 | 42.12 | ワイニーで野性的 |

In [ ]:
# 三大産地の座標
regions = {
    'Sidamo': (6.77, 38.50),
    'Yirgacheffe': (6.16, 38.20),
    'Harrar': (9.31, 42.12),
}

# 比較用：適地ではないエリア（アディスアベバ＝首都／ダナキル砂漠＝低地高温）
comparison = {
    'Addis Ababa (都市)': (9.03, 38.74),
    'Danakil (低地砂漠)': (14.24, 40.30),
}

results = []

for name, (lat, lon) in {**regions, **comparison}.items():
    point = ee.Geometry.Point([lon, lat])
    
    # 半径5km円内の平均
    buffer = point.buffer(5000)
    
    stats = ee.Image.cat([
        elevation.rename('elev'),
        temp_celsius,
        annual_precip,
        ndvi,
        suitability
    ]).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=buffer,
        scale=500,
        maxPixels=1e9
    ).getInfo()
    
    results.append({
        '地点': name,
        '標高(m)': round(stats.get('elev', 0), 0),
        '気温(℃)': round(stats.get('temp_c', 0), 1),
        '降水量(mm)': round(stats.get('precip_mm', 0), 0),
        'NDVI': round(stats.get('ndvi', 0), 2),
        '適地スコア': round(stats.get('suitability', 0), 2),
    })

df = pd.DataFrame(results)
print('=== 検証結果 ===')
df

### 検証の解釈

三大産地で適地スコアが高く、首都・砂漠地帯で低ければ、モデルが**既知の適地分布を正しく再現できている**と言える。  
→ このマップは「現状の」適地推定として信頼できる、と結論づけられる。

---
## 7. 適地スコアの分布

エチオピア全土でどれくらいの面積が「優良な適地」になっているかを集計する。

In [ ]:
# 適地スコアを4階級に分類
classified = ee.Image(0) \
    .where(suitability.gt(0.0).And(suitability.lte(0.3)), 1) \
    .where(suitability.gt(0.3).And(suitability.lte(0.5)), 2) \
    .where(suitability.gt(0.5).And(suitability.lte(0.7)), 3) \
    .where(suitability.gt(0.7), 4)

# 各クラスの面積（km²）
pixel_area = ee.Image.pixelArea().divide(1e6)  # m² → km²

areas = {}
labels = {1: '不適', 2: '限界的', 3: '良好', 4: '最適'}

for class_val, label in labels.items():
    area = pixel_area.updateMask(classified.eq(class_val)).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=ethiopia_geom,
        scale=1000,
        maxPixels=1e10
    ).getInfo()
    areas[label] = round(area.get('area', 0), 0)

# 結果表示
df_area = pd.DataFrame(list(areas.items()), columns=['区分', '面積(km²)'])
df_area['割合(%)'] = (df_area['面積(km²)'] / df_area['面積(km²)'].sum() * 100).round(1)
print('=== エチオピア国内のコーヒー適地分布 ===')
df_area

In [ ]:
# 棒グラフで可視化
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#f0f0f0', '#fff3a0', '#ffae42', '#d62828']
ax.bar(df_area['区分'], df_area['面積(km²)'], color=colors, edgecolor='black')
ax.set_ylabel('面積 (km²)', fontsize=12)
ax.set_title('エチオピアのコーヒー適地分布（現状）', fontsize=14, weight='bold')
for i, v in enumerate(df_area['面積(km²)']):
    ax.text(i, v + max(df_area['面積(km²)']) * 0.02, f'{v:,.0f}\n({df_area["割合(%)"][i]}%)',
            ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('outputs/suitability_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. 「コーヒー2050年問題」への接続（既存研究の引用）

本ノートブックでは現状の適地マッピングまでを自力で実装した。  
将来予測については、既存研究の結果を引用して考察する。

### 主要な既存研究

**Bunn et al. (2015)** *A bitter cup: climate change profile of global production of Arabica and Robusta coffee*  
→ 2050年までに、現在のアラビカ栽培地の **約50%が不適地化** すると予測。

**Moat et al. (2017)** *Resilience potential of the Ethiopian coffee sector under climate change*  
→ エチオピアでは、現在の栽培地の **39〜59%が2050年に不適地化** する一方、高標高地への拡大で**緩和可能**と指摘。

**Davis et al. (2019)** *High extinction risk for wild coffee species*  
→ 野生コーヒー種の **60%が絶滅危機**。エチオピアの遺伝子資源保全の重要性。

### 本研究との接続

本ノートブックで得た「現状の適地マップ」は、これら将来予測研究の **ベースライン** として機能する。  
今後、IPCCのSSPシナリオによる将来気温データを同じ閾値モデルに入力すれば、**簡易的な2050年予測**も可能。

---
## 9. まとめと考察

### 9-1. 実装で達成したこと

1. ✅ Google Earth Engineで4種類の衛星データ（標高・気温・降水量・NDVI）を統合
2. ✅ アラビカコーヒーの農学的適地条件を閾値モデルに落とし込み、適地スコアを算出
3. ✅ 三大産地（Sidamo・Yirgacheffe・Harrar）でモデルの妥当性を検証
4. ✅ エチオピア全土の適地分布を面積ベースで集計

### 9-2. 限界と今後の課題

- **閾値モデルの単純さ**：実際には土壌・斜面方位・霜害頻度なども効くため、機械学習による精緻化が望ましい
- **NDVIで「植生」しか見ていない**：コーヒー樹そのものの分類には、教師データを使った分類モデルが必要
- **将来予測**：IPCCシナリオ統合は今回のスコープ外。次フェーズの課題

### 9-3. スターバックス18年の経験との接続

私はスターバックスで18年間、エチオピア産シダモやイルガチェフェを扱ってきた。  
「あのカップに入っているコーヒーが、どこで、どのような環境で育っているか」を**衛星から俯瞰**できたことは、現場経験と科学的視点をつなぐ初めての体験だった。

コーヒー2050年問題は、**生産者・流通・消費者すべてに関わる課題**。  
現場で「価値を伝える」仕事をしてきた経験を、これからは**データで「価値を可視化する」仕事**として活かしていきたい。

---

## 📚 参考文献

- Bunn, C. et al. (2015). *A bitter cup: climate change profile of global production of Arabica and Robusta coffee.* Climatic Change, 129, 89–101.
- Moat, J. et al. (2017). *Resilience potential of the Ethiopian coffee sector under climate change.* Nature Plants, 3, 17081.
- Davis, A.P. et al. (2019). *High extinction risk for wild coffee species and implications for coffee sector sustainability.* Science Advances, 5, eaav3473.
- DaMatta, F.M. et al. (2007). *Ecophysiology of coffee growth and production.* Brazilian Journal of Plant Physiology, 19(4).
- ICO (International Coffee Organization) 各種レポート